In [ ]:
# !pip3 install seaborn
# !pip3 install matplotlib
# !pip3 install openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

In [ ]:
df_2024 = pd.read_excel("ttc-subway-delay-data-2024.xlsx", sheet_name="Subway")
df_2025 = pd.read_csv("ttc-subway-delay-data-since-2025.csv")

In [ ]:
df = pd.concat([df_2024, df_2025], ignore_index=True)
print(f"Combined dataset: {df.shape[0]} records, {df.shape[1]} columns")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
line_mapping = {
    # Line 1 — Yonge-University
    'YU': 'Line 1', 'YUS': 'Line 1', 'LINE 1': 'Line 1',
    # Line 2 — Bloor-Danforth
    'BD': 'Line 2', 'BLOOR DANFORTH': 'Line 2',
    # Line 4 — Sheppard
    'SHP': 'Line 4', 'SHEP': 'Line 4',
    # Line 3 — Scarborough RT (decommissioned 2023)
    'SRT': 'Line 3',
    # Multi-line: Line 1 / Line 2
    'YU/BD': 'Line 1/2', 'YU/ BD': 'Line 1/2', 'YU / BD': 'Line 1/2',
    'YU & BD': 'Line 1/2', 'YU/BD LINES': 'Line 1/2', 'YU -BD LINES': 'Line 1/2',
    'BD/YU': 'Line 1/2', 'BD/ YU': 'Line 1/2', 'BD / YU': 'Line 1/2',
    'YUS/BD': 'Line 1/2', 'YUS / BD': 'Line 1/2', 'YUS/ BD': 'Line 1/2',
    'BD/YUS': 'Line 1/2', 'BD/ YUS': 'Line 1/2', 'ONGE-UNIVERSITY AND BL': 'Line 1/2',
    # Multi-line: Line 1 / Line 2 / Line 4
    'YU/BD/SHP': 'Line 1/2/4', 'YUS/ BD/ SHP': 'Line 1/2/4',
    # Invalid — bus routes and non-subway entries
    '109 RANEE': None, '20 CLIFFSIDE': None, '29 DUFFERIN': None, 'TRACK LEVEL ACTIVITY': None,
}

In [ ]:
df['Line'] = df['Line'].map(line_mapping)
print(f"Unique Line values after maping: {df['Line'].dropna().nunique()}")
df['Line'].value_counts()


In [ ]:
records_before = len(df)
df = df.dropna(subset=['Line'])
print(f"Removed {records_before - len(df)} invalid records {records_before:,} -> {len(df):,}")


In [ ]:
print(df["Min Delay"].describe().round(2))


In [ ]:
# 25th and 75th percentiles -- define the middle 50% of the data
Q1 = df['Min Delay'].quantile(0.25)
Q3 = df['Min Delay'].quantile(0.75)

# IQR: spread of the middle 50%
IQR = Q3 - Q1

# Upper fence: standard threshold beyond which a value is an outlier
upper_fence = Q3 + 1.5 * IQR

In [ ]:
plt.figure(figsize=(6,6))

plt.boxplot(df["Min Delay"],
            vert=True,
            patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.6))

plt.title("Box Plot - Min Delay")
plt.ylabel("Minutes")
plt.show()

In [ ]:
df['min_delay_capped'] = df['Min Delay'].clip(upper=upper_fence)


In [ ]:
plt.figure(figsize=(6,6))

plt.boxplot(df['min_delay_capped'],
            vert=True, patch_artist=True,
            boxprops=dict(facecolor='steelblue',
            alpha=0.6))

plt.title('Box Plot - Min Delay', fontsize=13)
plt.ylabel('Minutes')
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['Min Delay'], bins=80, color='steelblue', alpha=0.7, edgecolor='white')
plt.axvline(df['Min Delay'].median(), color='red', linestyle='-', linewidth=1.5,
            label=f'Median ({df["Min Delay"].median():.0f} min)')
plt.axvline(df['Min Delay'].mean(), color='green', linestyle='--', linewidth=1.5,
            label=f'Mean ({df["Min Delay"].mean():.1f} min)')
plt.title('Histogram -- Min Delay', fontsize=13)
plt.xlabel('Minutes')
plt.ylabel('Count')
plt.legend(fontsize=9)
plt.xlim(left=0)
plt.tight_layout()
plt.show()

In [ ]:
# Histogram -- Min Delay
plt.figure(figsize=(8, 5))
plt.hist(df['min_delay_capped'], bins=80, color='steelblue', alpha=0.7, edgecolor='white')
plt.axvline(df['min_delay_capped'].median(), color='red', linestyle='-', linewidth=1.5,
            label=f'Median ({df["min_delay_capped"].median():.0f} min)')
plt.axvline(df['min_delay_capped'].mean(), color='green', linestyle='--', linewidth=1.5,
            label=f'Mean ({df["min_delay_capped"].mean():.1f} min)')
plt.title('Histogram -- Min Delay', fontsize=13)
plt.xlabel('Minutes')
plt.ylabel('Count')
plt.legend(fontsize=9)
plt.xlim(left=0)
plt.tight_layout()
plt.show()

In [ ]:
df = df.drop(columns=['_id', 'Vehicle', 'Bound'])
print(f"Remaining Columns  are: {df.columns.tolist()}")


In [ ]:
df['delay_occured'] = (df['Min Delay'] > 0).astype(int)

In [ ]:
print(df['delay_occured'].value_counts())

In [ ]:
print(df['delay_occured'].value_counts(normalize=True).mul(100).round(1).astype(str)+'%')

In [ ]:
df.head()

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

print(df['Date'].min())
print(df['Date'].max())

In [ ]:
df['hour'] = df['Time'].apply(lambda t: int(t.split(":")[0]))


In [ ]:
import requests
# Toronto coordinates
lat, lon = 43.651070, -79.347015

# Fetch hourly weather data from Open-Meteo for 2024 to 2026
url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    "&start_date=2024-01-01&end_date=2026-09-01"
    "&hourly=temperature_2m,precipitation,weathercode,windspeed_10m"
    "&timezone=America%2FToronto"
)

response = requests.get(url)
data = response.json()

In [ ]:
weather_df = pd.DataFrame({
    'datetime':      pd.to_datetime(data['hourly']['time']),
    'temperature':   data['hourly']['temperature_2m'],
    'precipitation': data['hourly']['precipitation'],
    'weathercode':   data['hourly']['weathercode'],
    'windspeed':     data['hourly']['windspeed_10m']
})

In [ ]:
print(weather_df.head(5))
print(df.head(5))

In [ ]:
weather_df['Date'] = weather_df['datetime'].dt.date
weather_df['hour'] = weather_df['datetime'].dt.hour

weather_df = weather_df.drop(columns=["datetime"])

weather_df.head(5)

In [ ]:
df['Date'] = pd.to_datetime(df['Date']).dt.date

df = df.merge(weather_df, on=["Date", "hour"], how="left")

print(df[["temperature", "precipitation", "weathercode", "windspeed"]].isnull().sum())


In [ ]:
day_mapping = {
    'Monday': 0,
    'Tuesday': 1, 
    'Wednesday': 2, 
    'Thursday': 3, 
    'Friday': 4, 
    'Saturday': 5, 
    'Sunday': 6, 
}

df['day_num'] = df['Day'].map(day_mapping)


In [ ]:
# !pip3 install scikit-learn

In [ ]:

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['station_code'] = le.fit_transform(df['Station'])

print(df[['Station', 'station_code']].drop_duplicates().head(10))

In [ ]:
df['line_num'] = le.fit_transform(df['Line'])

print(df[['line_num', 'Line']].drop_duplicates().head(10))

In [ ]:
# This follows the lesson; using all rows for this rate can make test scores optimistic.
delay_rate = df.groupby(['Station', 'hour'])['delay_occured'].mean()
delay_rate.head(100)

In [ ]:
df['delay_rate'] = df.set_index(['Station', 'hour']).index.map(delay_rate)

print(df[['Line', 'Station', 'Day', 'hour', 'delay_occured', 'delay_rate']].head(20))

In [ ]:
df_model = df.drop(columns=['Date', 'Time', 'Day', 'Station', 'Line', 'Code', 'Min Gap'])
df_model.head()

In [ ]:
# Features - TTC and weather combined
X = df_model[['hour', 'day_num', 'station_code', 'line_num', 'delay_rate',
              'temperature', 'precipitation', 'weathercode', 'windspeed']]

# Target/Labels
y = df_model['delay_occured']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model_rf.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report

y_pred = model_rf.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["No Delay", "Delay"]))

In [ ]:
# !pip3 install xgboost
# If macOS reports a missing libomp.dylib, run brew install libomp in Terminal.

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)

print(classification_report(y_test, y_pred_xgb, target_names=["No Delay", "Delay"]))

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train, y_train)

In [ ]:
y_pred_lr = lr_model.predict(X_test)

print(classification_report(y_test, y_pred_lr, target_names=["No Delay", "Delay"]))